# PaddleOCR with OpenVINO™

This demo shows how to run [PP-OCRv6](https://github.com/PaddlePaddle/PaddleOCR/blob/main/docs/version3.x/algorithm/PP-OCRv6/PP-OCRv6.en.md) text detection and recognition models with OpenVINO. [PaddleOCR](https://github.com/PaddlePaddle/PaddleOCR) is a multilingual OCR toolkit built on the PaddlePaddle deep learning framework. PP-OCRv6 is its latest generation of OCR models and comes in three tiers, tiny, small and medium. Each tier consists of a DB text detection model and a multilingual CTC text recognition model.

The notebook reads the official ONNX exports of the models from Hugging Face (for example, [`PaddlePaddle/PP-OCRv6_medium_det_onnx`](https://huggingface.co/PaddlePaddle/PP-OCRv6_medium_det_onnx) and [`PaddlePaddle/PP-OCRv6_medium_rec_onnx`](https://huggingface.co/PaddlePaddle/PP-OCRv6_medium_rec_onnx)) directly with OpenVINO, without a conversion step. Working pipeline of the PaddleOCR is as follows:

<img align='center' src= "https://raw.githubusercontent.com/yoyowz/classification/master/images/pipeline.png" alt="drawing" width="1000"/>

> **NOTE**: To use this notebook with a webcam, you need to run the notebook on a computer with a webcam. If you run the notebook on a server, the webcam will not work. You can still do inference on a video file.


#### Table of contents:

- [Imports](#Imports)
    - [Select inference device](#Select-inference-device)
    - [Models for PaddleOCR](#Models-for-PaddleOCR)
        - [Select the Model Tier](#Select-the-Model-Tier)
        - [Download the Model for Text **Detection**](#Download-the-Model-for-Text-**Detection**)
        - [Load the Model for Text **Detection**](#Load-the-Model-for-Text-**Detection**)
        - [Download the Model for Text **Recognition**](#Download-the-Model-for-Text-**Recognition**)
        - [Load the Model for Text **Recognition**](#Load-the-Model-for-Text-**Recognition**)
    - [Preprocessing Image Functions for Text Detection and Recognition](#Preprocessing-Image-Functions-for-Text-Detection-and-Recognition)
    - [Postprocessing Image for Text Detection](#Postprocessing-Image-for-Text-Detection)
    - [Main Processing Function for PaddleOCR](#Main-Processing-Function-for-PaddleOCR)
- [Run PaddleOCR on a Video Frame](#Run-PaddleOCR-on-a-Video-Frame)
- [Run Live PaddleOCR with OpenVINO](#Run-Live-PaddleOCR-with-OpenVINO)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/paddle-ocr-webcam/paddle-ocr-webcam.ipynb" />

In [ ]:
import urllib.request
from pathlib import Path

# Helpers come from `utils` of the cloned repository, otherwise from GitHub; local copies are overwritten
helper_files = ["notebook_utils.py"]
repo_utils = Path("../../utils")
in_repo = (repo_utils / "notebook_utils.py").is_file()
if in_repo:
    helper_contents = {helper_file: (repo_utils / helper_file).read_bytes() for helper_file in helper_files}
else:
    try:
        helper_contents = {}
        for helper_file in helper_files:
            url = f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{helper_file}"
            with urllib.request.urlopen(url, timeout=10) as response:
                helper_contents[helper_file] = response.read()
    except OSError as error:
        if any(not Path(helper_file).is_file() for helper_file in helper_files):
            raise
        print(f"Could not update helper files, using the existing copies: {error}")
        helper_contents = {}

for helper_file, content in helper_contents.items():
    Path(helper_file).write_bytes(content)

from notebook_utils import setup_notebook_venv

# Requirements installed below go to a notebook-specific virtual environment in the `.venv` folder
setup_notebook_venv()

In [ ]:
%pip install -q "openvino>=2026.4.1"
%pip install -q "pyclipper>=1.2.1" "shapely>=1.7.1" opencv-python tqdm "Pillow>=11.0" pyyaml ipywidgets

## Imports
[back to top ⬆️](#Table-of-contents:)


In [ ]:
import cv2
import numpy as np
import math
import time
import collections
from PIL import Image
from pathlib import Path

import ipywidgets as widgets
import openvino as ov
from IPython import display

In [3]:
# Import local modules

import notebook_utils as utils
import pre_post_processing as processing

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("paddle-ocr-webcam.ipynb")

### Select inference device
[back to top ⬆️](#Table-of-contents:)

select device from dropdown list for running inference using OpenVINO

In [ ]:
device = utils.device_widget()

device

### Models for PaddleOCR
[back to top ⬆️](#Table-of-contents:)

PaddleOCR includes two parts of deep learning models, text detection and text recognition. PP-OCRv6 provides them in three tiers that trade accuracy for speed:

| Tier | Text detection | Text recognition |
|---|---|---|
| tiny | [`PP-OCRv6_tiny_det_onnx`](https://huggingface.co/PaddlePaddle/PP-OCRv6_tiny_det_onnx) (1.7 MB) | [`PP-OCRv6_tiny_rec_onnx`](https://huggingface.co/PaddlePaddle/PP-OCRv6_tiny_rec_onnx) (4.3 MB) |
| small | [`PP-OCRv6_small_det_onnx`](https://huggingface.co/PaddlePaddle/PP-OCRv6_small_det_onnx) (9.4 MB) | [`PP-OCRv6_small_rec_onnx`](https://huggingface.co/PaddlePaddle/PP-OCRv6_small_rec_onnx) (20.2 MB) |
| medium | [`PP-OCRv6_medium_det_onnx`](https://huggingface.co/PaddlePaddle/PP-OCRv6_medium_det_onnx) (59.2 MB) | [`PP-OCRv6_medium_rec_onnx`](https://huggingface.co/PaddlePaddle/PP-OCRv6_medium_rec_onnx) (73.0 MB) |

The ONNX models and their `inference.yml` configuration files are downloaded from fixed revisions of the Hugging Face repositories and stored in the "model" folder. `inference.yml` provides the pre- and post-processing parameters of each model and the character dictionary of the recognition model.

Only a few lines of code are required to run the model. First, initialize the runtime for inference. Then, read the ONNX model with OpenVINO Runtime and compile it to the selected device.

#### Select the Model Tier
[back to top ⬆️](#Table-of-contents:)

The tier switches both the text detection and the text recognition model. **medium** is the default tier of PaddleOCR.

In [ ]:
model_tier = widgets.Dropdown(options=["tiny", "small", "medium"], value="medium", description="Model tier:")

model_tier

In [ ]:
# Hugging Face revisions of the ONNX model repositories used in this notebook.
model_revisions = {
    "PP-OCRv6_tiny_det_onnx": "2ba1506c0380b8f0b03dd142459aac66d4421f6c",
    "PP-OCRv6_tiny_rec_onnx": "2612ab37152ae0a677521bae4e1e3d4fb4cf7c30",
    "PP-OCRv6_small_det_onnx": "28fe5895c24fd108c19eb3e8479f4ab385fbfc62",
    "PP-OCRv6_small_rec_onnx": "b8f84f0b80c529de40b4fbb3544b84fa7233a513",
    "PP-OCRv6_medium_det_onnx": "61323801669c338b7891481ec7bac61ce31b576a",
    "PP-OCRv6_medium_rec_onnx": "50c7eacafc52fa7bcf4194e8cd08e46f8558504b",
}


def model_file_url(model_name: str, file_name: str) -> str:
    return f"https://huggingface.co/PaddlePaddle/{model_name}/resolve/{model_revisions[model_name]}/{file_name}"


# Define the function to download text detection and recognition model files.


def run_model_download(model_url: str, model_file_path: Path) -> None:
    """
    Download pre-trained model files from Hugging Face.

    Parameters:
        model_url: URL link to the pre-trained model file.
        model_file_path: File path to store the downloaded model file.
    """
    if model_file_path.is_file():
        print("Model already exists")
    else:
        print("Downloading the pre-trained model... May take a while...")
        utils.download_file(model_url, model_file_path.name, model_file_path.parent)
        print(f"Model Downloaded to {model_file_path}")

#### Download the Model for Text **Detection**
[back to top ⬆️](#Table-of-contents:)


In [ ]:
# A directory where the model will be downloaded.
det_model_name = f"PP-OCRv6_{model_tier.value}_det_onnx"
det_model_file_path = Path(f"model/{det_model_name}/inference.onnx")
det_config_file_path = Path(f"model/{det_model_name}/inference.yml")

run_model_download(model_file_url(det_model_name, "inference.onnx"), det_model_file_path)
run_model_download(model_file_url(det_model_name, "inference.yml"), det_config_file_path)

#### Load the Model for Text **Detection**
[back to top ⬆️](#Table-of-contents:)

CPU runs the text detection model with dynamic input shapes: each frame is resized with its aspect ratio kept, so that the shorter side is at least 736 pixels and both sides are multiples of 32.

NPU requires static input shapes, and GPU also runs faster with them. For these devices, and for AUTO, which may select them, the model is reshaped to `[1, 3, 1024, 1024]`. Each frame is letterboxed into this square: it is resized with its aspect ratio kept and padded,     instead of being stretched.

In [ ]:
# Initialize OpenVINO Runtime for text detection.
core = ov.Core()

# CPU uses dynamic input shapes; GPU, NPU and AUTO use static ones.
use_static_shapes = device.value != "CPU"
det_static_size = 1024

# Pre- and post-processing parameters of the text detection model.
det_config = processing.load_det_config(det_config_file_path)

det_model = core.read_model(model=det_model_file_path)

if use_static_shapes:
    det_model.reshape({det_model.input(0): ov.PartialShape([1, 3, det_static_size, det_static_size])})

det_compiled_model = core.compile_model(model=det_model, device_name=device.value)

# Get input and output nodes for text detection.
det_input_layer = det_compiled_model.input(0)
det_output_layer = det_compiled_model.output(0)

#### Download the Model for Text **Recognition**
[back to top ⬆️](#Table-of-contents:)


In [ ]:
rec_model_name = f"PP-OCRv6_{model_tier.value}_rec_onnx"
rec_model_file_path = Path(f"model/{rec_model_name}/inference.onnx")
rec_config_file_path = Path(f"model/{rec_model_name}/inference.yml")

run_model_download(model_file_url(rec_model_name, "inference.onnx"), rec_model_file_path)
run_model_download(model_file_url(rec_model_name, "inference.yml"), rec_config_file_path)

#### Load the Model for Text **Recognition**
[back to top ⬆️](#Table-of-contents:)

Input to the text recognition model refers to detected bounding boxes with different image sizes. Each box image is resized to a height of 48 pixels with its aspect ratio kept. Hence:

1. CPU keeps a dynamic recognition width by assigning `-1` to the input width dimension and recognizes up to 6 text lines in a batch.
2. GPU, NPU and AUTO use static shapes: the model is compiled for a few fixed widths (320, 640, 960, 1280 and 1920 pixels). Each box image goes to the smallest width that fits it and is padded on the right. A single fixed width would squeeze or cut long text lines.

In [ ]:
# Recognition input shape [3, 48, 320] and the character dictionary of the text recognition model.
rec_config = processing.load_rec_config(rec_config_file_path)
rec_image_height = rec_config["image_shape"][1]
rec_widths = [320, 640, 960, 1280, 1920]

# Read the model from a file.
rec_model = core.read_model(model=rec_model_file_path)

if use_static_shapes:
    rec_batch_num = 1
    # One compiled model per static input width.
    rec_compiled_models = {}
    for rec_width in rec_widths:
        static_rec_model = rec_model.clone()
        static_rec_model.reshape({static_rec_model.input(0): ov.PartialShape([rec_batch_num, 3, rec_image_height, rec_width])})
        rec_compiled_models[rec_width] = core.compile_model(model=static_rec_model, device_name=device.value)
else:
    rec_batch_num = 6
    # Assign dynamic shapes to every input layer on the last dimension.
    for input_layer in rec_model.inputs:
        input_shape = input_layer.partial_shape
        input_shape[3] = -1
        rec_model.reshape({input_layer: input_shape})
    rec_compiled_models = {None: core.compile_model(model=rec_model, device_name=device.value)}

# Decoder of the text recognition model output.
postprocess_op = processing.CTCLabelDecode(rec_config["character_list"])

### Preprocessing Image Functions for Text Detection and Recognition
[back to top ⬆️](#Table-of-contents:)


Define preprocessing functions for text detection and recognition:
1. Preprocessing for text detection: resize (dynamic shapes) or letterbox (static shapes) input images and normalize them with the mean and standard deviation from `inference.yml`.
2. Preprocessing for text recognition: resize detected box images to a height of 48 pixels, normalize them and pad them to the same width (for example, `(3, 48, 320)` size) for easy batching in inference.

In [10]:
# Preprocess for text detection.
def image_preprocess(input_image):
    """
    Preprocess input image for text detection

    Parameters:
        input_image: input image in BGR format
    Returns:
        input tensor for the text detection model and (height, width) of the image area in it
    """
    normalize_params = (det_config["mean"], det_config["std"], det_config["scale"])
    if use_static_shapes:
        img, (resize_h, resize_w) = processing.letterbox(input_image, det_static_size)
        # The padded area stays zero after normalization.
        padded_img = np.zeros((det_static_size, det_static_size, 3), dtype=np.float32)
        padded_img[:resize_h, :resize_w] = processing.NormalizeImage(img, *normalize_params)
        img = padded_img
    else:
        img, _ = processing.DetResizeForTest(input_image, det_config["limit_side_len"], det_config["limit_type"], det_config["max_side_limit"])
        resize_h, resize_w = img.shape[:2]
        img = processing.NormalizeImage(img, *normalize_params)
    img = np.expand_dims(np.transpose(img, [2, 0, 1]), 0)
    return img.astype(np.float32), (resize_h, resize_w)

In [ ]:
# Preprocess for text recognition.
def resize_norm_img(img, img_width):
    """
    Resize input image for text recognition

    Parameters:
        img: bounding box image from text detection
        img_width: width of the text recognition model input
    """
    img_channels, img_height = rec_config["image_shape"][:2]
    assert img_channels == img.shape[2]
    h, w = img.shape[:2]
    resized_w = min(img_width, int(math.ceil(img_height * w / float(h))))
    resized_image = cv2.resize(img, (resized_w, img_height))
    resized_image = resized_image.astype("float32")
    resized_image = resized_image.transpose((2, 0, 1)) / 255
    resized_image -= 0.5
    resized_image /= 0.5
    padding_im = np.zeros((img_channels, img_height, img_width), dtype=np.float32)
    padding_im[:, :, 0:resized_w] = resized_image
    return padding_im


def prep_for_rec(dt_boxes, frame):
    """
    Preprocessing of the detected bounding boxes for text recognition

    Parameters:
        dt_boxes: detected bounding boxes from text detection
        frame: original input frame
    """
    img_crop_list = [processing.get_rotate_crop_image(frame, box.copy()) for box in dt_boxes]
    img_num = len(img_crop_list)
    # Calculate the aspect ratio of all text bars.
    width_list = [img.shape[1] / float(img.shape[0]) for img in img_crop_list]

    # Sorting can speed up the recognition process.
    indices = np.argsort(np.array(width_list))
    return img_crop_list, img_num, indices


def batch_text_box(img_crop_list, img_num, indices, beg_img_no, batch_num):
    """
    Batch for text recognition

    Parameters:
        img_crop_list: processed detected bounding box images
        img_num: number of bounding boxes from text detection
        indices: sorting for bounding boxes to speed up text recognition
        beg_img_no: the beginning number of bounding boxes for each batch of text recognition inference
        batch_num: number of images for each batch
    Returns:
        batch of images and the static input width to use (None for dynamic shapes)
    """
    img_height, default_width = rec_config["image_shape"][1:]
    end_img_no = min(img_num, beg_img_no + batch_num)
    max_wh_ratio = default_width / img_height
    for ino in range(beg_img_no, end_img_no):
        h, w = img_crop_list[indices[ino]].shape[0:2]
        max_wh_ratio = max(max_wh_ratio, w * 1.0 / h)
    img_width = int(img_height * max_wh_ratio)
    rec_width = None
    if use_static_shapes:
        # The smallest static width that fits the text line; longer lines are squeezed into the widest one.
        rec_width = next((width for width in rec_widths if width >= img_width), rec_widths[-1])
        img_width = rec_width
    norm_img_batch = [resize_norm_img(img_crop_list[indices[ino]], img_width)[np.newaxis, :] for ino in range(beg_img_no, end_img_no)]
    return np.concatenate(norm_img_batch), rec_width

### Postprocessing Image for Text Detection
[back to top ⬆️](#Table-of-contents:)


In [ ]:
def post_processing_detection(frame, det_results, image_shape):
    """
    Postprocess the results from text detection into bounding boxes

    Parameters:
        frame: input image
        det_results: inference results from text detection model
        image_shape: (height, width) of the image area in the text detection model input
    """
    resize_h, resize_w = image_shape
    # Drop the padded area of a letterboxed input; the remaining map covers the whole frame.
    pred = det_results[0, 0, :resize_h, :resize_w]
    segmentation = pred > det_config["thresh"]
    src_h, src_w = frame.shape[:2]
    dt_boxes, _ = processing.boxes_from_bitmap(
        pred,
        segmentation,
        src_w,
        src_h,
        box_thresh=det_config["box_thresh"],
        unclip_ratio=det_config["unclip_ratio"],
        max_candidates=det_config["max_candidates"],
    )
    return processing.filter_tag_det_res(dt_boxes, frame.shape)

### Main Processing Function for PaddleOCR
[back to top ⬆️](#Table-of-contents:)


The `run_ocr` function recognizes text in one frame: it runs text detection, crops the detected text boxes and runs text recognition on them.

Run `paddleOCR` function in different operations, either a webcam or a video file. See the list of procedures below:

1. Create a video player to play with target fps (`utils.VideoPlayer`).
2. Prepare a set of frames for text detection and recognition.
3. Run AI inference for both text detection and recognition.
4. Visualize the results.

In [ ]:
# Download a font for printing OCR results.
font_path = Path("fonts/simfang.ttf")

if not font_path.exists():
    utils.download_file(
        url="https://raw.githubusercontent.com/Halfish/lstm-ctc-ocr/master/fonts/simfang.ttf",
        directory="fonts",
    )

In [ ]:
def run_ocr(frame):
    """
    Run text detection and recognition on one frame

    Parameters:
        frame: input image in BGR format
    Returns:
        detected text boxes, recognized texts and their confidence levels
    """
    # Preprocess the image for text detection.
    test_image, image_shape = image_preprocess(frame)

    # Perform the inference step.
    det_results = det_compiled_model([test_image])[det_output_layer]

    # Postprocessing for Paddle Detection.
    dt_boxes = post_processing_detection(frame, det_results, image_shape)

    # Preprocess detection results for recognition.
    dt_boxes = processing.sorted_boxes(dt_boxes)
    img_crop_list, img_num, indices = prep_for_rec(dt_boxes, frame)

    # For storing recognition results, include two parts:
    # txts are the recognized text results, scores are the recognition confidence level.
    rec_res = [("", 0.0)] * img_num
    for beg_img_no in range(0, img_num, rec_batch_num):
        # Recognition starts from here.
        norm_img_batch, rec_width = batch_text_box(img_crop_list, img_num, indices, beg_img_no, rec_batch_num)

        # Run inference for text recognition.
        rec_compiled_model = rec_compiled_models[rec_width]
        rec_results = rec_compiled_model([norm_img_batch])[rec_compiled_model.output(0)]

        # Postprocessing recognition results.
        rec_result = postprocess_op(rec_results)
        for rno in range(len(rec_result)):
            rec_res[indices[beg_img_no + rno]] = rec_result[rno]
    txts = [text for text, _ in rec_res]
    scores = [score for _, score in rec_res]
    return dt_boxes, txts, scores


def run_paddle_ocr(source=0, flip=False, use_popup=False, skip_first_frames=0):
    """
    Main function to run the paddleOCR inference:
    1. Create a video player to play with target fps (utils.VideoPlayer).
    2. Prepare a set of frames for text detection and recognition.
    3. Run AI inference for both text detection and recognition.
    4. Visualize the results.

    Parameters:
        source: The webcam number to feed the video stream with primary webcam set to "0", or the video path.
        flip: To be used by VideoPlayer function for flipping capture image.
        use_popup: False for showing encoded frames over this notebook, True for creating a popup window.
        skip_first_frames: Number of frames to skip at the beginning of the video.
    """
    # Create a video player to play with target fps.
    player = None
    try:
        player = utils.VideoPlayer(source=source, flip=flip, fps=30, skip_first_frames=skip_first_frames)
        # Start video capturing.
        player.start()
        if use_popup:
            title = "Press ESC to Exit"
            cv2.namedWindow(winname=title, flags=cv2.WINDOW_GUI_NORMAL | cv2.WINDOW_AUTOSIZE)

        processing_times = collections.deque()
        while True:
            # Grab the frame.
            frame = player.next()
            if frame is None:
                print("Source ended")
                break
            # If the frame is larger than full HD, reduce size to improve the performance.
            scale = 1280 / max(frame.shape)
            if scale < 1:
                frame = cv2.resize(
                    src=frame,
                    dsize=None,
                    fx=scale,
                    fy=scale,
                    interpolation=cv2.INTER_AREA,
                )

            # Measure processing time for text detection and recognition.
            start_time = time.time()
            boxes, txts, scores = run_ocr(frame)
            stop_time = time.time()

            processing_times.append(stop_time - start_time)
            # Use processing times from last 200 frames.
            if len(processing_times) > 200:
                processing_times.popleft()
            processing_time = np.mean(processing_times) * 1000

            image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            # Draw text recognition results beside the image.
            draw_img = processing.draw_ocr_box_txt(image, boxes, txts, scores, drop_score=0.5, font_path=str(font_path))

            # Visualize the PaddleOCR results.
            f_height, f_width = draw_img.shape[:2]
            fps = 1000 / processing_time
            cv2.putText(
                img=draw_img,
                text=f"OCR time: {processing_time:.1f}ms ({fps:.1f} FPS)",
                org=(20, 40),
                fontFace=cv2.FONT_HERSHEY_COMPLEX,
                fontScale=f_width / 1000,
                color=(0, 0, 255),
                thickness=1,
                lineType=cv2.LINE_AA,
            )

            # Use this workaround if there is flickering.
            if use_popup:
                draw_img = cv2.cvtColor(draw_img, cv2.COLOR_RGB2BGR)
                cv2.imshow(winname=title, mat=draw_img)
                key = cv2.waitKey(1)
                # escape = 27
                if key == 27:
                    break
            else:
                # Encode numpy array to jpg.
                draw_img = cv2.cvtColor(draw_img, cv2.COLOR_RGB2BGR)
                _, encoded_img = cv2.imencode(ext=".jpg", img=draw_img, params=[cv2.IMWRITE_JPEG_QUALITY, 100])
                # Create an IPython image.
                i = display.Image(data=encoded_img)
                # Display the image in this notebook.
                display.clear_output(wait=True)
                display.display(i)

    # ctrl-c
    except KeyboardInterrupt:
        print("Interrupted")
    # any different error
    except RuntimeError as e:
        print(e)
    finally:
        if player is not None:
            # Stop capturing.
            player.stop()
        if use_popup:
            cv2.destroyAllWindows()

## Run PaddleOCR on a Video Frame
[back to top ⬆️](#Table-of-contents:)

Before the live demo, run OCR on the first frame of the test video and print the recognized text lines. Only text lines with a confidence level of at least 0.5 are printed and drawn.

In [ ]:
video_file = Path("test.mp4")

if not video_file.exists():
    utils.download_file("https://raw.githubusercontent.com/yoyowz/classification/master/images/test.mp4")

capture = cv2.VideoCapture(str(video_file))
_, sample_frame = capture.read()
capture.release()

boxes, txts, scores = run_ocr(sample_frame)

print("Recognized text:")
for text, score in zip(txts, scores):
    if score >= 0.5:
        print(f"{score:.2f}  {text}")

sample_image = Image.fromarray(cv2.cvtColor(sample_frame, cv2.COLOR_BGR2RGB))
Image.fromarray(processing.draw_ocr_box_txt(sample_image, boxes, txts, scores, drop_score=0.5, font_path=str(font_path)))

## Run Live PaddleOCR with OpenVINO
[back to top ⬆️](#Table-of-contents:)


Use a webcam as the video input. By default, the primary webcam is set with `source=0`. If you have multiple webcams, each one will be assigned a consecutive number starting at 0. Set `flip=True` when using a front-facing camera. Some web browsers, especially Mozilla Firefox, may cause flickering. If you experience flickering, set `use_popup=True`. 

> **NOTE**: Popup mode may not work if you run this notebook on a remote computer.

If you do not have a webcam, you can still run this demo with a video file. Any [format supported by OpenCV](https://docs.opencv.org/4.5.1/dd/d43/tutorial_py_video_display.html) will work.

Run live PaddleOCR:

In [ ]:
USE_WEBCAM = False

cam_id = 0

source = cam_id if USE_WEBCAM else str(video_file)

run_paddle_ocr(source, flip=False, use_popup=False, skip_first_frames=0)